In [ ]:
from dotenv import load_dotenv
from ollama import Client
import os
import pymupdf
import json
from pydantic import BaseModel
load_dotenv()

OLLAMA_API_KEY = os.environ.get("OLLAMA_CLOUD_API_KEY")
PDF_PATH = "../data/JohnatanWrightResume.pdf"
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

class ResumeData(BaseModel):
    first_name: str
    last_name: str
    email: str
    phone: str
    experience_years: int
    skills: list[str]
    job_title: str
    experience: list[str]
    education: list[str]
    summary: str

def _get_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def extract_structured_data(text):
    schema = ResumeData.model_json_schema()
    response = client.chat(
        model="gpt-oss:120b",
        messages=[{"role": "user", "content": f"""
                   You are a data extraction engine.Extract resume data and return only valid JSON from this text: {text}.
                   Return ONLY valid JSON that strictly conforms to this schema: {schema}.
                   Rules:
                   - No markdown
                   - No comments
                   - No extra fields
                   - Use null for missing values
                   """}],
        format="json",
        options={"temperature": 0}
    )
    return response
    

text_from_pdf = _get_text_from_pdf(PDF_PATH)
structured_data =  extract_structured_data(text_from_pdf)
structured_json = structured_data['message']['content']



{
  "first_name": "Johnatan",
  "last_name": "Wright",
  "email": null,
  "phone": null,
  "experience_years": 12,
  "skills": [
    "ASP.Net MVC",
    "Git",
    "REST",
    "Visual Studio",
    "UI/UX Design",
    "NHibernate",
    "Entity Framework",
    "Distributed Systems",
    "Static Analysis",
    "Unit Testing",
    "Continuous Integration",
    "Real Time Data Applications",
    "Verbal and Written Communication",
    "Customer Engagement",
    "Matrixed Team Environment",
    "Quick Learner"
  ],
  "job_title": "Software Development Engineer",
  "experience": [
    "Software Development Engineer at Accenture Federal Services (2018-2022): Designed and developed customer-facing applications using .NET Framework/Core, created mission-critical system with 30% faster response times, maintained database schema and API design reducing code maintenance time by 50%, collaborated with team to deliver production code on time and within budget.",
    "Senior Software Engineer at Booz A